<img src="images/top_banner_ML1_HW1.png" style="width: 100%;">

# *Binary Classification of Wines for Bottling Process Automation using K-Nearest Neighbors*

**COSCI221 - Machine Learning 1**

**Kenneth T. Co, PhD**
  
*Homework 1*

**<u>MSDS 2026 Learning Team 8, Term 2,</u>**

- Francis Erdey M. Capati
   
- Kevin Ansel S. Dy

- Jan Paolo V. Moreno

- Mia Cielo G. Oliveros

[I. Background](##I.-Background)

[II. Wine Type Classification Data Set](##II.-Wine-Type-Classification-Data-Set)

[III. Exploratory Data Analysis](##III.-Exploratory-Data-Analysis)

[IV. Applying K-Nearest Neighbors to Classify Wine Type](##IV.-Applying-K-Nearest-Neighbors-to-Classify-Wine-Type)

[V. References](##References)

## I. Background
 
<p align="justify">Part of the wine manufacturing process is wine bottling. In this step, a bulk storage tank containing wine is 
transferred to a bottling factory and placed in wine bottles (Mohais et. al, 2012, p. 39).</p>
 
<p align="justify">A bottling factory contains bottling lines which are machines that are directly connected to the tank, and are used to transfer the wine into glass bottles (Mohais et. al, 2012, p. 39). For most companies, it is impractical to have individual machines exclusively for each type of wine. As such, one machine is used at different times to bottle different types of wine. The process of switching from one wine type to another type is called <i>wine changeover</i> (Mohais et. al, 2012, p. 41).</p>

Machines also change the bottles to be used depending on the type of wine. Darker bottles are used for red wine to prevent oxidation and degradation of aromas and flavors, while clearer bottles are used for white wine since they are generally consumed more quickly (Boroli, 2024).

Given the dependency of the wine bottle to be used on the type of wine, it is important for the machines to correctly identify the type of wine that it is working with. 
                                                                                       
### Problem: 
Create a model that predicts the wine type by classifying as red wine or white wine based on its chemical properties.

## II. Wine Type Classification Data Set

### A. Import necessary libraries

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split

### B. Load dataset

In [2]:
df = pd.read_csv('wine_quality_merged.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'wine_quality_merged.csv'

### C. Initial information about the dataset

The Wine Type Classification Dataset contains 6,497 samples of red and white wines. This includes 13 features describing their chemical properties and quality.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

### Features and Target

In [ ]:
conda install -c conda-forge rdkit

In [ ]:
!pip install py3dmol nglview ipywidgets
!jupyter-nbextension enable nglview --py --sys-prefix

In [14]:
!pip install py3Dmol ipywidgets
!jupyter-nbextension enable --py widgetsnbextension --sys-prefix


Defaulting to user installation because normal site-packages is not writeable
/bin/bash: line 1: jupyter-nbextension: command not found


In [15]:
import py3Dmol
from IPython.display import HTML

smiles = "CCO"
view = py3Dmol.view(width=400, height=400)
view.addModel(smiles, "smi")
view.setStyle({'stick': {}})
view.zoomTo()
HTML(view._make_html())  # forces inline rendering


In [17]:
import py3Dmol
from IPython.display import display, HTML

# Key wine-related molecules (SMILES + chemical role)
molecules = {
    "Ethanol (Alcohol)": {
        "smiles": "CCO",
        "role": "Primary alcohol from fermentation; affects aroma, viscosity, and volatility."
    },
    "Tartaric Acid (Fixed Acidity)": {
        "smiles": "OC(C(O)=O)C(O)=O",
        "role": "Principal organic acid in grapes; controls pH, freshness, and color stability."
    },
    "Acetic Acid (Volatile Acidity)": {
        "smiles": "CC(=O)O",
        "role": "Formed by spoilage bacteria; high levels cause vinegar-like aroma."
    },
    "Sulfur Dioxide (Preservative)": {
        "smiles": "O=S=O",
        "role": "Antimicrobial and antioxidant agent; extends shelf life and prevents oxidation."
    },
    "Resveratrol (Polyphenol)": {
        "smiles": "C1=CC(=CC=C1C=CC2=CC(O)=CC(O)=C2)O",
        "role": "Phenolic compound abundant in red wine; provides antioxidant and health benefits."
    }
}

# Function to render a molecule as an interactive 3D model
def show_molecule(name, smiles, role):
    view = py3Dmol.view(width=320, height=320)
    view.addModel(smiles, "smi")
    view.setStyle({'stick': {}})
    view.zoomTo()
    html = f"""
    <div style='display:inline-block; text-align:center; margin:15px;'>
        <h4 style='font-family:sans-serif;'>{name}</h4>
        {view._make_html()}
        <p style='font-size:13px; max-width:280px; text-align:justify;'>{role}</p>
    </div>
    """
    display(HTML(html))

# Display all molecules one after another
for name, data in molecules.items():
    show_molecule(name, data["smiles"], data["role"])


In [6]:
import nglview
import ipywidgets

In [7]:
!jupyter-nbextension enable --py widgetsnbextension --sys-prefix

/bin/bash: line 1: jupyter-nbextension: command not found


In [13]:
from IPython.display import display, HTML
import py3Dmol

smiles = "CCO"
view = py3Dmol.view(width=400, height=400)
view.addModel(smiles, "smi")
view.setStyle({'stick': {}})
view.zoomTo()
display(HTML(view._make_html()))


#### 1. Acidity/complexity (Fixed acidity, volatile acidity, citric acid, pH)

##### Fixed acidity
Description: The fixed acid content (tartaric, malic, and lactic acid) contributes to the wine’s taste and stability, balancing sweetness and sourness.

##### Volatile acidity
Description: Volatile acids, such as acetic acid, can cause an unpleasant vinegar taste at higher concentrations.

##### Citric acid
Description: Adds freshness and enhances the flavor balance (and clarity of color) of the wine, intensifies the fruitiness in the wine's aromatic profile, improving its overall taste and preventing ferric hazes.

##### pH
Description: The measure of acidity; determines wine stability and shelf life.

#### 2. Fermentation and Texture/Body (Alcohols, Residual sugars, Density)

##### Alcohol
Description: Alcohol content in the wine, ethanol, determines body, warmth, aroma intensity and flavor. It acts as a solvent, extracting color, tannins, and flavor compounds from the grape skins during the winemaking process.

##### Residual sugar
Description: Ripe grapes contain natural sugars (glucose and fructose) that are converted to alcohol and carbon dioxide during fermentation. 

##### Density
Description: The density of wine, influenced by alcohol and sugar content. According to Michlovský, 2023, Red Wine and White Wine density are comparable at 0.9912 to 1.0138 g/cm3.

#### 3. Preservation/stability (Free sulphate, Total sulphate, Sulphates)

##### Free sulfur dioxide
Description:  The portion of sulfur dioxide that remains active and available to perform its preservative functions. 

##### Total sulfur dioxide
Description: Composed of free and bound sulfur dioxide. Bound SO₂: The portion that has reacted with other compounds in the wine (like acetaldehyde) and is no longer effective against oxidation and microbes. Greatly influenced by pH.

##### Sulphates (Should be sulphites?)
Description: Sulphites bind with oxygen, protecting the wine from oxidation (browning and loss of aroma). All wines contain some sulphites, as a natural byproduct of the fermentation process. 

#### 4. Unique Characteristic/Context (Chlorides, Type, Quality)

##### Chlorides
Description: The levels of chloride ions can help determine a wine's geographical origin and the grape varieties used. Wines from coastal regions tend to have higher chloride levels, making it more salty.

##### Type
Description: Indicates whether the wine is Red or White. White wine is produced by fermenting the clarified juice of green or yellow grape varieties without skin contact, yielding lower phenolic and tannin content, whereas red wine undergoes fermentation with the skins of dark grape varieties, promoting the extraction of anthocyanins, tannins, and other polyphenolic compounds that enhance color, structure, and antioxidant capacity.

##### Quality
Description: 

In [ ]:
df['quality'].unique()

In [ ]:
df['type'].unique()

In [ ]:
df.nunique() #Count of unique values per column

In [ ]:
df.isna().any() #Checking if there is NaN

## III. Exploratory Data Analysis

### Pairplots per Feature Category

#### 1. Pairplot: Acidity/complexity

Based on the pairplot below, the type of wine is not easily distinguishable from its acidity or complexity, as red and white wines share overlapping levels of fixed acidity, volatile acidity, citric acid, and pH.

In [ ]:
sns.pairplot(df, vars=['fixed acidity', 'volatile acidity', 'citric acid', 'pH'], hue='type', palette={'red': '#7B0323', 'white': '#F1F285'})

#### 2. Pairplot: Fermentation and Texture/Body

In [ ]:
sns.pairplot(df, vars=['alcohol', 'residual sugar', 'density'], hue='type', palette={'red': '#7B0323', 'white': '#F1F285'})

#### 3. Pairplot: Preservation/stability

In terms of preservation and stability, red and white wines cannot be easily differentiated, as they exhibit nearly the same levels of free sulfur dioxide, total sulfur dioxide, and sulphates.

In [ ]:
sns.pairplot(df, vars=['free sulfur dioxide', 'total sulfur dioxide', 'sulphates'], hue='type', palette={'red': '#7B0323', 'white': '#F1F285'})

#### 4. Pairplot: Unique Characteristic/Context

When comparing chlorides and quality, red and white wines appear alike, offering no clear distinction between the two types.

In [ ]:
sns.pairplot(df, vars=['chlorides', 'quality'], hue='type', palette={'red': '#7B0323', 'white': '#F1F285'})

Overall, the analysis suggests that red and white wines cannot be clearly distinguished based on their chemical properties. Across all examined features such as acidity, complexity, fermentation, texture, body, preservation, and stability, both types exhibit overlapping levels of fixed acidity, volatile acidity, citric acid, pH, alcohol, residual sugar, density, free sulfur dioxide, total sulfur dioxide, and sulphates. Even when considering chlorides and quality, the similarities persist, indicating that these characteristics alone are insufficient to differentiate one wine type from the other.

## IV. Applying K-Nearest Neighbors to Classify Wine Type

In [ ]:
df

In [ ]:
# Step 1: get the data
 
X = df.drop(columns=['quality','type'])
y = df['type']
 
# Step 2: split the data
 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)
 
# Step 3: Instantiate the model, define the learning parameter, say number of neighbor
 
knn = KNeighborsClassifier(n_neighbors=3)
 
# Step 4: Fit the model using the training data and testing target
 
knn.fit(X_train, y_train)

In [ ]:
# Step 5: Test the model
 
y_test_pred = knn.predict(X_test)
print("Test set predictions:\n {}".format(y_test_pred))

In [ ]:
# Step 6: Determine accuracy
 
print("Test set score: {:.4f}".format(knn.score(X_test, y_test)))

In [ ]:
X = df.drop(columns=['quality', 'type'])  
y = df['type']  # target

lahat_training = pd.DataFrame()
lahat_test = pd.DataFrame()

for seedN in range(1, 20, 1):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=seedN
    )

    training_accuracy = []
    test_accuracy = []
    neighbors_settings = range(1, 50, 1)  # try n_neighbors from 1 to 50

    for n_neighbors in neighbors_settings:
        clf = KNeighborsClassifier(n_neighbors=n_neighbors)
        clf.fit(X_train, y_train)

        training_accuracy.append(clf.score(X_train, y_train))
        test_accuracy.append(clf.score(X_test, y_test))

    lahat_training[seedN] = training_accuracy
    lahat_test[seedN] = test_accuracy

#### Monte Carlo Cross Validation

In [ ]:
plt.errorbar(neighbors_settings, lahat_training.mean(axis=1),
             yerr=lahat_training.std(axis=1)/2, label="training accuracy")
plt.errorbar(neighbors_settings, lahat_test.mean(axis=1),
             yerr=lahat_test.std(axis=1)/6, label="test accuracy")
plt.ylabel("Accuracy")
plt.xlabel("n_neighbors")
plt.legend()

In [ ]:
print("Best test set mean accuracy: {:.2f}%".format(lahat_test.mean(axis = 1).max() * 100))
print("For k = {}".format(lahat_test.mean(axis = 1).argmax() + 1))


## V. References

[1] Mohais, A., Schellenberg, S., Ibrahimov, M., Wagner, N., & Michalewicz, Z. (2011). An Evolutionary Approach to practical constraints in Scheduling: A Case-Study of the Wine Bottling Problem. In Springer eBooks (pp. 31–58). https://doi.org/10.1007/978-3-642-23424-8_2

[2] Cholette, S. (2011). Postponement Practices in the Wine Industry: Comparison of Adaptation and Attitudes between California and Bordeaux.

[3] (2024, September 24). Why Do Wine Bottles Have Different Colors?- Boroli - Vino Barolo. Boroli - Vino Barolo. https://www.boroli.it/en/complete-guide-to-wine-bottle-colors/
    
[4] Moccia, Luigi. (2013). Operational Research in the Wine Supply Chain. INFOR: Information Systems and Operational Research. 51. 53-63. 10.3138/infor.51.2.53. 

<img src="images/bottom_banner_ML1_HW1.png" style="width: 100%;">